In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("🟢 Step 1 executed")
    return {"step1": "done", "input": state["input"]}


def step_2(state: CrashState) -> CrashState:
    print("🟡 Step 2, now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(10)  # Simulate long-running hang
    return {"step2": "done"}


def step_3(state: CrashState) -> CrashState:
    print("🟢 Step 3 executed")
    return {"step3": "done"}

In [ ]:
# 3. Build the graph
builder = StateGraph(CrashState)

builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
graph

In [24]:
try:
    print("🟢 Running graph: Please manually interrupt during Step 2...")
    graph.invoke(
        {"input": "start"},
        config={"configurable": {"thread_id": "thread-1"}}
    )
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

🟢 Running graph: Please manually interrupt during Step 2...
🟢 Step 1 executed
🟡 Step 2, now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [25]:
graph.get_state({"configurable":{"thread_id":"thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac426-e4bd-6b8e-800f-1683d0cad0c0'}}, metadata={'source': 'loop', 'step': 15, 'parents': {}}, created_at='2026-09-09T11:34:34.347982+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac426-e4aa-61c5-800e-2ee081058f73'}}, tasks=(PregelTask(id='22990a72-eb7d-5218-14fe-adb2bb35b0da', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [26]:
list(graph.get_state_history({"configurable":{"thread_id":"thread-1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac426-e4bd-6b8e-800f-1683d0cad0c0'}}, metadata={'source': 'loop', 'step': 15, 'parents': {}}, created_at='2026-09-09T11:34:34.347982+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac426-e4aa-61c5-800e-2ee081058f73'}}, tasks=(PregelTask(id='22990a72-eb7d-5218-14fe-adb2bb35b0da', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac426-e4aa-61c5-800e-2ee081058f73'}}, metadata={'source': 'loop', 'step': 14, 'parents': {}}, created_at='2026-09-09T11:34:34.339954+00

In [27]:
#6. Re-run to show fault-tolerant resume
print("\n Re-running the graph to demostrate fault tolerance..")
final_state = graph.invoke(None,config={"configurable":{"thread_id":"thread-1"}})
print("\n Final State:", final_state)


 Re-running the graph to demostrate fault tolerance..
🟡 Step 2, now manually interrupt from the notebook toolbar (STOP button)
🟢 Step 3 executed

 Final State: {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


In [28]:
list(graph.get_state_history({"configurable":{"thread_id":"thread-1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac429-1df2-6005-8011-7dd317b4d7c0'}}, metadata={'source': 'loop', 'step': 17, 'parents': {}}, created_at='2026-09-09T11:35:34.033389+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac429-1dee-66cc-8010-2c71e97e6aa3'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac429-1dee-66cc-8010-2c71e97e6aa3'}}, metadata={'source': 'loop', 'step': 16, 'parents': {}}, created_at='2026-09-09T11:35:34.031876+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac426-e4bd-6b8e-800f-1683d0cad0c0'}}, tasks=(PregelTask(id=